In [1]:
# Imported the required libraries. Boto3 essentially enables the local notebook to pull from the S3 bucket.

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import boto3

BUCKET = "cancer-source-data"


Matrix shape: (487, 16136)
cancer_type
LUAD    99
BRCA    99
COAD    98
PRAD    97
KIRC    94
Name: count, dtype: int64


/Users/eesha/cancer-classifier/venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Done — plots uploaded to s3://cancer-source-data/output/plots/


In [ ]:
# Built the mutation matrix.

df = pd.read_parquet(f"s3://{BUCKET}/processed/clean_maf/")

labels = df[['Tumor_Sample_Barcode', 'cancer_type']].drop_duplicates()
labels = labels.set_index('Tumor_Sample_Barcode')['cancer_type']
assert df.groupby('Tumor_Sample_Barcode')['cancer_type'].nunique().max() == 1

matrix = pd.crosstab(df['Tumor_Sample_Barcode'], df['Hugo_Symbol'])
matrix = (matrix > 0).astype('int8')
matrix['cancer_type'] = labels
matrix.to_parquet(f"s3://{BUCKET}/processed/matrix/mutation_matrix.parquet")

print("Matrix shape:", matrix.shape)
print(matrix['cancer_type'].value_counts())

gene_cols = matrix.columns.drop('cancer_type')


In [ ]:
# Did some basic EDA.

plt.figure()
matrix['cancer_type'].value_counts().plot(kind='bar', title='Patients per cancer type')
plt.tight_layout()
plt.savefig('class_distribution.png')
plt.close()

freq = matrix.groupby('cancer_type')[gene_cols].mean()
top_genes = freq.max(axis=0).nlargest(20).index

plt.figure()
sns.heatmap(freq[top_genes], cmap='viridis')
plt.title('Top 20 gene mutation frequency by cancer type')
plt.tight_layout()
plt.savefig('top_genes_heatmap.png')
plt.close()

matrix['burden'] = matrix[gene_cols].sum(axis=1)

plt.figure()
sns.boxplot(data=matrix, x='cancer_type', y='burden')
plt.title('Mutations per patient by cancer type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('mutation_burden.png')
plt.close()


In [ ]:
# Uploaded EDA plots to S3.

s3 = boto3.client('s3')
for f in ['class_distribution.png', 'top_genes_heatmap.png', 'mutation_burden.png']:
    s3.upload_file(f, BUCKET, f'output/plots/{f}')

print("Done — plots uploaded to s3://%s/output/plots/" % BUCKET)
